In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Point Python to your src folder so it can find config.py
# Robust src path — works whether run manually or via papermill
_src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from config import *

print("Packages loaded successfully")
print(f"Raw data path : {RAW_PATH}")
print(f"Output path   : {OUT_PATH}")

In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_FILE = PROJECT_ROOT / RAW_PATH

df = pd.read_excel(RAW_FILE, sheet_name=RAW_SHEET)

print("=" * 50)
print("SHAPE")
print(f"{df.shape[0]} rows  {df.shape[1]} columns")

print("\nCOLUMNS")
for col in df.columns:
    print(col)

print("\nDATATYPES")
print(df.dtypes.to_string())

SHAPE
127319 rows  11 columns

COLUMNS
year
month
Brand
Range
Size
Division
Material
Final Channel
qty
net_sales
Portal

DATATYPES
year               int64
month              int64
Brand             object
Range             object
Size              object
Division          object
Material          object
Final Channel     object
qty                int64
net_sales        float64
Portal            object


In [10]:
print("MISSING VALUES")
print(df.isnull().sum().to_string())

print("\nNEGATIVE OR ZERO QTY")
print(f"  qty <= 0  : {(df['qty'] <= 0).sum()} rows")
print(f"  net_sales <= 0 : {(df['net_sales'] <= 0).sum()} rows")

print("\nYEAR DISTRIBUTION")
print(df['year'].value_counts().sort_index().to_string())

print("\nFINAL CHANNEL DISTRIBUTION")
print(df['Final Channel'].value_counts().to_string())

MISSING VALUES
year             0
month            0
Brand            0
Range            1
Size             0
Division         0
Material         0
Final Channel    0
qty              0
net_sales        0
Portal           0

NEGATIVE OR ZERO QTY
  qty <= 0  : 0 rows
  net_sales <= 0 : 0 rows

YEAR DISTRIBUTION
year
2023    28911
2024    48752
2025    49656

FINAL CHANNEL DISTRIBUTION
Final Channel
EC    127319


In [11]:
print("FULL YEAR DISTRIBUTION")
print(df['year'].value_counts().sort_index().to_string())

print("\nDIVISION DISTRIBUTION")
print(df['Division'].value_counts().to_string())

print("\nSIZE DISTRIBUTION")
print(df['Size'].value_counts().to_string())

print("\nSEGMENT INVENTORY (Division x Portal x Size)")
print("How many rows each segment has:\n")
inventory = (df.groupby(['Division', 'Portal', 'Size'])
               .size()
               .reset_index(name='row_count')
               .sort_values('row_count', ascending=False))
print(inventory.to_string(index=False))
print(f"\nTotal segments: {len(inventory)}")

FULL YEAR DISTRIBUTION
year
2023    28911
2024    48752
2025    49656

DIVISION DISTRIBUTION
Division
BP    61729
HL    31439
SL    19973
DF    13297
BS      881

SIZE DISTRIBUTION
Size
Single    62610
LARGE     15268
CABIN     14788
MEDIUM    14289
DFT        7991
DF         5306
SO3        4466
SO2        2601

SEGMENT INVENTORY (Division x Portal x Size)
How many rows each segment has:

Division   Portal   Size  row_count
      BP       MP Single      40210
      BP   Others Single       8416
      BP      D2C Single       6722
      DF       MP    DFT       6040
      HL       MP MEDIUM       5391
      HL       MP  CABIN       5337
      SL       MP  LARGE       4944
      HL       MP  LARGE       4545
      SL       MP  CABIN       3459
      DF       MP     DF       3328
      SL       MP MEDIUM       3131
      BP Flipkart Single       2227
      BP   Myntra Single       2076
      BP   Amazon Single       1848
      HL Flipkart MEDIUM       1494
      HL Flipkart  CABIN       

In [12]:
# Strip whitespace from every string column — fixes the SO3 / 'SO3 ' problem
for col in ['Division', 'Size', 'Final Channel'] + ([RANGE_COL] if RANGE_COL in df.columns else []):
    df[col] = df[col].astype(str).str.strip()

# Apply business filters
df_clean = df[~df['year'].isin(IGNORE_YEARS)].copy()
df_clean = df_clean[df_clean['Final Channel'] == CHANNEL].copy()

# Confirm the phantom segment is gone
print("AFTER CLEANING")
print(f"Rows remaining : {len(df_clean):,}")
print(f"Years remaining: {sorted(df_clean['year'].unique())}")

print("\nSEGMENT INVENTORY AFTER CLEANING")
inventory_clean = (df_clean.groupby(['Division', 'Portal', 'Size'])
                            .size()
                            .reset_index(name='row_count')
                            .sort_values(['Division', 'Size']))
print(inventory_clean.to_string(index=False))
print(f"\nTotal segments: {len(inventory_clean)}")

# Check SO3 is now clean
print("\nSO3 check (should be exactly one entry):")
print(inventory_clean[inventory_clean['Size'].str.contains('SO3')])

AFTER CLEANING
Rows remaining : 98,408
Years remaining: [np.int64(2024), np.int64(2025)]

SEGMENT INVENTORY AFTER CLEANING
Division   Portal   Size  row_count
      BP   Amazon Single        956
      BP  Blinkit Single        230
      BP      D2C Single       5527
      BP Flipkart Single       1566
      BP       MP Single      32993
      BP   Myntra Single       1139
      BP   Others Single       5240
      BS   Amazon Single         33
      BS  Blinkit Single          2
      BS      D2C Single         16
      BS Flipkart Single         15
      BS       MP Single        572
      BS   Myntra Single         14
      BS   Others Single         73
      DF   Amazon     DF        292
      DF  Blinkit     DF         70
      DF      D2C     DF        200
      DF Flipkart     DF        253
      DF       MP     DF       2841
      DF   Myntra     DF         49
      DF   Others     DF        580
      DF   Amazon    DFT        206
      DF      D2C    DFT        185
      DF Flip

In [13]:
df_clean['sale_date'] = pd.to_datetime(
    dict(year=df_clean['year'], month=df_clean['month'], day=1)
)

print("DATE RANGE")
print(f"  From : {df_clean['sale_date'].min().strftime('%b %Y')}")
print(f"  To   : {df_clean['sale_date'].max().strftime('%b %Y')}")
print(f"  Months: {df_clean['sale_date'].nunique()}")

print("\nQTY AND NET_SALES SANITY CHECK")
print(f"  Total qty sold    : {df_clean['qty'].sum():,.0f} units")
print(f"  Total net sales   : ₹{df_clean['net_sales'].sum():,.0f}")
print(f"  Overall avg ASP   : ₹{(df_clean['net_sales'].sum() / df_clean['qty'].sum()):,.0f}")

print("\nQTY PER DIVISION (2024+2025 combined)")
div_summary = (df_clean.groupby('Division')
                       .agg(qty=('qty','sum'), net_sales=('net_sales','sum'))
                       .assign(ASP=lambda x: x['net_sales']/x['qty'])
                       .sort_values('qty', ascending=False))
div_summary['qty']       = div_summary['qty'].map('{:,.0f}'.format)
div_summary['net_sales'] = div_summary['net_sales'].map('₹{:,.0f}'.format)
div_summary['ASP']       = div_summary['ASP'].map('₹{:,.0f}'.format)
print(div_summary.to_string())

# Save clean dataframe for notebook 02
os.makedirs(OUT_PATH, exist_ok=True)
df_clean.to_csv(f'{OUT_PATH}01_clean_sales.csv', index=False)
print(f"\nSaved: {OUT_PATH}01_clean_sales.csv")

DATE RANGE
  From : Jan 2024
  To   : Dec 2025
  Months: 24

QTY AND NET_SALES SANITY CHECK
  Total qty sold    : 8,479,787 units
  Total net sales   : ₹15,481,424,115
  Overall avg ASP   : ₹1,826

QTY PER DIVISION (2024+2025 combined)
                qty       net_sales     ASP
Division                                   
BP        3,674,392  ₹2,506,225,767    ₹682
HL        2,887,217  ₹9,046,592,437  ₹3,133
SL          998,173  ₹3,057,491,081  ₹3,063
DF          892,559    ₹809,996,817    ₹907
BS           27,446     ₹61,118,013  ₹2,227

Saved: data/outputs/EC/01_clean_sales.csv
